# Turtle SS27 GPU artifact build

This notebook builds `frontend/app/generated-data.json` on a Google Colab GPU. It uses 100% visual ranking with a strict internal garment-body colour gate (displayed images remain unchanged), prints exact per-batch progress, validates SS27 image coverage, and saves the finished artifact back to Google Drive.

Before running, upload the **current local project folder** to `MyDrive/turtle-colab/turtle`. The required data is about 9.3 GB. You may omit `.git`, `.venv`, `node_modules`, and `frontend/.next`, but keep `DATA/raw`, the two XLSB workbooks, `DATA/processed`, `backend`, `frontend`, `Makefile`, and `tmp/real-data-converted` if available. Do not clone an older GitHub revision because it may still contain the removed SAM 2 code.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/turtle-colab/turtle')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/turtle-colab/artifacts')
LOCAL_PROJECT = Path('/content/turtle')
BATCH_SIZE = 16

assert DRIVE_PROJECT.is_dir(), f'Upload the current project to {DRIVE_PROJECT}'
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Drive project:', DRIVE_PROJECT)
print('Drive output:', DRIVE_OUTPUT)

## Copy the project to Colab local storage

Thousands of small image reads are much faster from `/content` than directly from mounted Drive. This cell replaces only the ephemeral `/content/turtle` copy; it does not modify the Drive source.

In [ ]:
import shutil
import subprocess

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)
LOCAL_PROJECT.mkdir(parents=True)

subprocess.run([
    'rsync', '-a',
    '--exclude=.git',
    '--exclude=.venv',
    '--exclude=node_modules',
    '--exclude=frontend/.next',
    f'{DRIVE_PROJECT}/', f'{LOCAL_PROJECT}/',
], check=True)
print('Copied project to', LOCAL_PROJECT)

## Verify GPU, code revision, inputs, and ID/image mapping

In [ ]:
import csv
import os

subprocess.run(['nvidia-smi'], check=True)

artifact_source = LOCAL_PROJECT / 'backend/src/fashion_matching/artifact_vision.py'
cli_source = LOCAL_PROJECT / 'backend/src/data_pipeline/prepare_real_data.py'
assert artifact_source.is_file()
assert cli_source.is_file()
assert 'garment-body CIELAB' in artifact_source.read_text(), 'Current visual-only colour-gate code was not uploaded'
assert '--verbose' in cli_source.read_text(), 'Verbose progress code was not uploaded'

raw = LOCAL_PROJECT / 'DATA/raw'
processed = LOCAL_PROJECT / 'DATA/processed'
upcoming_images = raw / 'upcoming_ss27_matched_images'
historical_images = raw / 'historical_matched_images'
required = [
    raw / 'LAST SEASONES ORDERING & SALE THRU DATA.xlsb',
    raw / 'SEG WISE SS27 MASTER SHEET TILL.xlsb',
    processed / 'upcoming_cleaned.csv',
    processed / 'historical_cleaned.csv',
    upcoming_images,
    historical_images,
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required inputs: {missing}'

extensions = {'.jpg', '.jpeg', '.png', '.webp'}
image_ids = {p.stem.upper() for p in upcoming_images.iterdir() if p.suffix.lower() in extensions}
with (processed / 'upcoming_cleaned.csv').open(encoding='utf-8-sig', newline='') as handle:
    upcoming_rows = list(csv.DictReader(handle))
product_ids = {row['product_id'].upper() for row in upcoming_rows}

print('Upcoming rows:', len(upcoming_rows))
print('Unique upcoming IDs:', len(product_ids))
print('Upcoming images:', len(image_ids))
print('Exact ID/image matches:', len(product_ids & image_ids))
print('Missing images:', len(product_ids - image_ids))
print('Extra images:', len(image_ids - product_ids))
assert len(product_ids) == 5550
assert image_ids == product_ids, 'Upcoming cleaned IDs and image filenames are not a 1:1 match'

## Install system and Python dependencies

LibreOffice is needed only when the cached XLSX conversions are absent or older than the XLSB sources.

In [ ]:
import sys

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'libreoffice'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(LOCAL_PROJECT / 'backend/requirements-ml.txt'),
    '-r', str(LOCAL_PROJECT / 'backend/requirements-fashion-matching.txt'),
], check=True)
print('Dependencies installed')

## Build the artifact on CUDA

Keep this tab connected. Each completed batch prints an exact processed count, percentage, elapsed time, ETA, encoded count, and skipped count. The full log is also saved to `/content/turtle-vision-build.log`.

In [ ]:
import datetime as dt

LOG_PATH = Path('/content/turtle-vision-build.log')
env = os.environ.copy()
env.update({
    'PYTHONPATH': str(LOCAL_PROJECT / 'backend/src'),
    'FASHION_MATCHING_DEVICE': 'cuda',
    'FASHION_MATCHING_BATCH_SIZE': str(BATCH_SIZE),
    'FASHION_MAX_IMAGE_BYTES': '16777216',
    'HF_HOME': '/content/huggingface-cache',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONUNBUFFERED': '1',
})
command = [
    sys.executable, '-m', 'data_pipeline.prepare_real_data',
    '--with-vision', '--verbose', '--batch-size', str(BATCH_SIZE),
]
print('Started:', dt.datetime.now().isoformat(timespec='seconds'))
print('Command:', ' '.join(command))

with LOG_PATH.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command,
        cwd=LOCAL_PROJECT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()

assert return_code == 0, f'Artifact build failed with exit code {return_code}; inspect {LOG_PATH}'
print('Finished:', dt.datetime.now().isoformat(timespec='seconds'))

## Validate the completed JSON

In [ ]:
import json

ARTIFACT = LOCAL_PROJECT / 'frontend/app/generated-data.json'
assert ARTIFACT.is_file(), f'Missing artifact: {ARTIFACT}'
with ARTIFACT.open(encoding='utf-8') as handle:
    artifact = json.load(handle)

meta = artifact['meta']
vision = meta['visionModel']
upcoming = artifact['upcoming']
historical = artifact['historical']
results = {
    'artifact_bytes': ARTIFACT.stat().st_size,
    'historical_records': len(historical),
    'upcoming_records': len(upcoming),
    'historical_image_coverage': meta.get('historicalImageCoverage'),
    'upcoming_image_coverage': meta.get('upcomingImageCoverage'),
    'historical_visual_coverage': vision.get('historicalCoverage'),
    'upcoming_visual_coverage': vision.get('upcomingCoverage'),
    'upcoming_visual_features': sum(bool(item.get('hasVisualFeature')) for item in upcoming),
    'model_id': vision.get('modelId'),
    'device': vision.get('device'),
}
print(json.dumps(results, indent=2))

assert len(upcoming) == 5550
assert meta.get('upcomingImageCoverage') == 5550
assert vision.get('upcomingCoverage') == 5550
assert results['upcoming_visual_features'] == 5550
assert vision.get('device') == 'cuda'
appearance = (vision.get('reranker') or {}).get('appearance') or {}
assert 'segmentation' not in appearance, 'Unexpected masking/segmentation metadata found'
assert meta['model'].get('visualOnlyRanking') is True
assert meta['model'].get('attributeWeight') == 0
assert meta['model'].get('visualWeight') == 1
assert (vision.get('reranker') or {}).get('sameItemTypeConstraint') is True
assert appearance.get('colourDescriptor', {}).get('fullImage') is False
assert appearance.get('colourGate', {}).get('enabled') is True
assert appearance.get('colourGate', {}).get('maximumDistance') == 0.2
print('Artifact validation passed')

## Save the artifact, log, checksum, and validation summary to Drive

In [ ]:
import hashlib

timestamp = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
versioned_artifact = DRIVE_OUTPUT / f'generated-data-ss27-{timestamp}.json'
canonical_artifact = DRIVE_OUTPUT / 'generated-data.json'
drive_log = DRIVE_OUTPUT / f'turtle-vision-build-{timestamp}.log'
summary_path = DRIVE_OUTPUT / f'turtle-vision-validation-{timestamp}.json'

shutil.copy2(ARTIFACT, versioned_artifact)
shutil.copy2(ARTIFACT, canonical_artifact)
shutil.copy2(LOG_PATH, drive_log)
checksum = hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()
validation = {**results, 'sha256': checksum, 'created_at': dt.datetime.now().isoformat()}
summary_path.write_text(json.dumps(validation, indent=2), encoding='utf-8')

print('Saved artifact:', versioned_artifact)
print('Canonical artifact:', canonical_artifact)
print('Saved log:', drive_log)
print('Saved validation:', summary_path)
print('SHA-256:', checksum)

## Optional direct browser download

The Drive copy is safer. Run this only if you also want the artifact downloaded through the browser.

In [ ]:
# from google.colab import files
# files.download(str(ARTIFACT))

After downloading or syncing the artifact to the Mac, replace `frontend/app/generated-data.json`, validate its checksum and coverage, and restart the backend so it loads the new model artifact.